# 01 - Perfilado inicial
Dataset crudo: INEGI EDR 2023 (DEFUN23.dbf), 74 variables, ~formato DBF.
Filtro objetivo: `Tipo_defun == 3` (Suicidio / Lesion autoinfligida).
Perfilado con pandas puro (sin ydata-profiling, por compatibilidad con Python 3.14+).

In [1]:
import pandas as pd
import sys
sys.path.append('../src')
from cleaning_utils import load_dbf, normalize_columns, null_summary, dtype_summary, duplicate_report, profiling_report, special_code_report


## 1. Cargar catalogos (tablas pequenas)
Necesarios para traducir codigos a etiquetas (entidad, municipio, causa CIE-10, etc.)

In [2]:
cat_geo = load_dbf('../data/raw/CATEMLDE23.dbf')      # Cve_ent, Cve_mun, Cve_loc, Nom_loc
cat_causa = load_dbf('../data/raw/CATMINDE.dbf')       # Cve, Descrip (CIE-10 detallado)
cat_listamex = load_dbf('../data/raw/LISTAMEX.dbf')    # Cve, Descrip (lista mexicana)
cat_capgpo = load_dbf('../data/raw/CAPGPO.dbf')        # Cap, Gpo, Descrip
cat_parentesco = load_dbf('../data/raw/PARENTESCO.dbf')

print('Catalogo geografico:', cat_geo.shape)
print('Catalogo causa (CIE-10):', cat_causa.shape)
print('Catalogo lista mexicana:', cat_listamex.shape)


Catalogo geografico: (28283, 4)
Catalogo causa (CIE-10): (4119, 2)
Catalogo lista mexicana: (422, 2)


## 2. Cargar tabla principal DEFUN23.dbf
131 MB, 74 columnas. Puede tardar 1-2 min en cargar.

In [3]:
df_raw = load_dbf('../data/raw/DEFUN23.dbf')
df_raw = normalize_columns(df_raw)  # TIPO_DEFUN -> Tipo_defun, etc.
print(f'Total de defunciones registradas 2023: {len(df_raw):,}')
df_raw.head()


Total de defunciones registradas 2023: 799,869


,Ent_regis,Mun_regis,Tloc_regis,Loc_regis,Ent_resid,Mun_resid,Tloc_resid,Loc_resid,Ent_ocurr,Mun_ocurr,...,Complicaro,Dia_cert,Mes_cert,Anio_cert,Maternas,Ent_ocules,Mun_ocules,Loc_ocules,Razon_m,Dis_re_oax
0,01,001,15,0001,32,044,5,0001,01,001,...,9,18,12,2022,,88,888,8888,NaN,999
1,01,001,15,0001,01,001,15,0001,01,001,...,9,12,12,2022,,88,888,8888,NaN,999
2,01,001,15,0001,01,001,15,0001,01,001,...,9,17,12,2022,,88,888,8888,NaN,999
3,01,001,15,0001,01,001,15,0001,01,001,...,9,1,1,2023,,88,888,8888,NaN,999
4,01,001,15,0001,14,053,1,7777,01,001,...,8,22,12,2022,,88,888,8888,NaN,999


## 3. Perfilado general (antes de filtrar)

In [ ]:
profiling_report(df_raw)


## 4. Explorar variable clave: Tipo_defun
1=Accidente, 2=Homicidio, 3=Suicidio, 4=Enfermedad, 5=Intervencion legal, 9=Se ignora

In [ ]:
df_raw['Tipo_defun'].value_counts(dropna=False).sort_index()


## 5. Filtrar universo de suicidio
Regla: Tipo_defun == 3. Verificar contra cifra oficial INEGI (~9,085 casos en 2023, 10.8% de 84,118 causas externas).

In [4]:
df_suicidio = df_raw[df_raw['Tipo_defun'] == 3].copy()
print(f'Registros de suicidio filtrados: {len(df_suicidio):,}')
print('Cifra oficial INEGI (nota tecnica EDR 2023): ~9,085 casos (10.8% de 84,118 causas externas)')


Registros de suicidio filtrados: 9,072
Cifra oficial INEGI (nota tecnica EDR 2023): ~9,085 casos (10.8% de 84,118 causas externas)


## 6. Perfilado del subconjunto de suicidio

In [ ]:
profiling_report(df_suicidio)


## 7. Exploracion de catalogos relevantes en el subconjunto
Sexo, edad, entidad, mes de ocurrencia, escolaridad, sitio de ocurrencia.

In [ ]:
df_suicidio['Sexo'].value_counts(dropna=False)


In [ ]:
df_suicidio['Ent_ocurr'].value_counts(dropna=False).head(15)


## 7b. Codigos de "no especificado" ocultos (no detectados como NaN)
Revisar valores como 8, 9, 88, 99, 997-999 en variables clave, que funcionalmente son nulos pero pandas no los detecta como tal.

In [5]:
reporte_codigos = special_code_report(df_suicidio)
reporte_codigos


,columna,codigo,conteo,pct_del_total
0,Ocurr_trab,9,2330,25.68
1,Derechohab,99,2312,25.49
2,Asist_medi,9,1701,18.75
3,Ocupacion,998,1539,16.96
4,Lugar_ocur,9,1353,14.91
5,Conindig,9,907,10.00
6,Lengua,9,712,7.85
7,Usonecrops,8,691,7.62
8,Afromex,9,684,7.54
9,Sitio_ocur,99,671,7.40


## 8. Hallazgos del perfilado
_Documentar aqui los problemas encontrados (nulos, codigos 'no especificado', inconsistencias) y trasladarlos a docs/methodology.md y docs/quality_report.md_

In [ ]:
## 8. Hallazgos del perfilado

### Validación de volumen
- Registros filtrados por Tipo_defun == 3 (suicidio): 9,072
- Cifra oficial INEGI (Nota Técnica EDR 2023): ~9,085
- Diferencia: 13 casos (0.14%) — dentro de margen aceptable. Filtro validado.

### Nulos estructurales (NaN detectados por pandas)
- Razon_m: 100% nulo — esperado, es variable exclusiva de defunciones maternas,
  no aplica a suicidios. No requiere tratamiento.
- Resto de columnas: 0% NaN explícito.

### Nulos ocultos (códigos categóricos de "no especificado"/"se ignora")
No detectados por pandas como NaN, pero funcionalmente lo son. Hallazgos >15%:
- Ocurr_trab (código 9): 25.68% — se ignora si ocurrió durante el trabajo.
- Derechohab (código 99): 25.49% — afiliación a salud no especificada.
- Asist_medi (código 9): 18.75% — se ignora si hubo atención médica previa.
- Ocupacion (códigos 998+999): 23.4% combinado — ocupación no especificada/no aplica.
- Lugar_ocur (código 9): 14.91% — se ignora el espacio físico del hecho.

Variables demográficas centrales, en cambio, están casi completas:
- Sexo: 0.02% no especificado (2 de 9,072 casos).
- Edad_agru: 1.03% no especificado.

### Duplicados
- 0 registros duplicados exactos sobre 9,072 filas.

### Decisión metodológica para 02_cleaning.ipynb
- Los códigos de "no especificado"/"se ignora" se recodificarán explícitamente
  a NaN (no se imputan) para reflejar honestamente la incompletitud del dato
  administrativo, siguiendo principio FAIR de transparencia.
- Variables con >20% de no especificado (Ocurr_trab, Derechohab) se marcarán
  en el diccionario de datos como "uso con precaución en análisis" en vez de
  excluirlas del dataset.
- Variables demográficas centrales (Sexo, Edad, entidad) se consideran
  confiables para análisis por su bajo nivel de no especificado.